# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 4d4261f2


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [6]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.

    Args:
        todos: List of todo items, each with 'title' and optional 'description'

    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.

    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)

    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.

    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"

    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [7]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [8]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [9]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.

    Args:
        path: Directory path to list (default: current directory)

    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"

    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")

    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.

    Args:
        path: Path to the file to read

    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).

    Args:
        path: Path to the file to write
        content: Content to write to the file

    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.

    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text

    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"

    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"

    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/denismcphillips/code/AIE9/07_Deep_Agents/workspace


In [10]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] 2_week_exercise_schedule.txt (1202 bytes)
[FILE] alex_comprehensive_wellness_program.md (8877 bytes)
[FILE] alex_exercise_plan.md (3365 bytes)
[FILE] alex_mindfulness_plan.md (9264 bytes)
[FILE] alex_nutrition_plan.md (6020 bytes)
[FILE] calming_elements.txt (672 bytes)
[FILE] comprehensive-morning-energy-guide.md (10119 bytes)
[FILE] comprehensive_morning_routine_guide.md (15948 bytes)
[FILE] cooldown_routine.txt (606 bytes)
[FILE] energizing_morning_routine_guide.md (13159 bytes)
[FILE] meal_plan_week_1.txt (3235 bytes)
[FILE] meal_plan_week_2.txt (2875 bytes)
[FILE] morning-energy-routine-guide.md (15342 bytes)
[DIR] morning_exercise_research
[FILE] morning_routine_guide.md (36433 bytes)
[FILE] morning_routine_research_summary.md (2980 bytes)
[FILE] my_sleep_improvement_plan.md (5854 bytes)
[DIR] nutrition
[FILE] nutritional_highlights.txt (802 bytes)
[FILE] personalized_sleep_improvement_plan.md (7581 bytes)
[FILE] prep_tips.txt (1039 bytes)
[DIR]

In [11]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] morning_routine_research_summary.md (2975 bytes)
[FILE] sleep_notes.md (242 bytes)


In [12]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [13]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/denismcphillips/code/AIE9/07_Deep_Agents/workspace


In [16]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Summary

I've created a comprehensive, evidence-based sleep improvement plan specifically tailored to address your three main issues:

### **Key Features of Your Plan:**

1. **Progressive 6-Week Implementation** - Gradual changes to ensure sustainable adoption
2. **Addresses Your Specific Issues:**
   - **Inconsistent bedtime:** Establishes consistent 7:00 AM wake time and 10:30 PM target bedtime
   - **Phone use in bed:** Progressive elimination with practical alternatives
   - **Morning tiredness:** Morning light exposure, sleep environment optimization

3. **Immediate Action Items** you can start tonight:
   - Set 7:00 AM alarm (consistency is key)
   - Move phone charger out of bedroom
   - Adjust bedroom temperature to 67°F

4. **Built-in Progress Tracking** with daily metrics and weekly check-ins

5. **Troubleshooting Guide** for common challenges you might face

### **Expected Results:**
- **Week 1:** Initial adjustment, improved wake time consistency
- **Week

In [14]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


Workspace contents:
  [FILE] comprehensive-morning-energy-guide.md (10119 bytes)
  [FILE] morning-energy-routine-guide.md (15342 bytes)
  [FILE] morning_routine_guide.md (36433 bytes)
  [FILE] morning_routine_research_summary.md (2980 bytes)
  [FILE] my_sleep_improvement_plan.md (5854 bytes)
  [FILE] personalized_sleep_improvement_plan.md (7581 bytes)
  [DIR] research/
  [FILE] sleep_improvement_quick_guide.md (4196 bytes)
  [FILE] sleep_improvement_research.md (16319 bytes)


---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
Todo lists can be helpful when planning responses for queries that contain multiple steps or require more thinking (i.e. the sleep routine plan in the notebook). However, for queries that require fewer steps this can be overkill (i.e. "What is the capital of France?"). Todos can offer good visibility to the steps an agent is taking to answer a query, but we should consider whether the query merits the additional context window space, costs, and added latency when deciding to use todos.

Todo items should be granular enough that they can reasonably stand alone as their own task (i.e. "research sleep improvement methods") but should not be too granular that too much context is added to the deep agent / the context added is overkill compared to what the model knows (i.e. "research what the circadian rhythm is"). 

If the agent creates todos but never completes them, some possibilities of why this happens could be that the agent has lost track of the todo during a long task, the todo was determined to no longer be relevant to the task, or the context window was reached without the todo getting completed.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
User conditions for safety should be in the prompt and should never be offloaded to ensure that the agent's responses don't give dangerous advice in its responses. These should be in long-term memory as well to ensure that these are not lost.

The large health document should be persisted in files, unless the information is deemed to be outdated - in which case it should be updated or replaced. It would be best to use retrieval to pull relevant information from this large document so that the entire file doesn't need to be read every time it's referenced.

User metrics should also be persisted in files or long-term memory, but these can be updated to only keep the most recent / relevant metrics as the number of metrics grows. Older metrics can be summarized and persisted if they are deemed relevant.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [34]:
### YOUR CODE HERE ###

# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide

# Note: We are building the agent in step 3; using this step to clear the todo store.
TODO_STORE.clear()

# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")
@tool
def read_wellness_guide() -> str:
    """Read the Health Wellness Guide from the data folder.

    Returns:
        The full contents of the HealthWellnessGuide.txt file
    """
    guide_path = Path("data/HealthWellnessGuide.txt")
    if not guide_path.exists():
        return "Error: HealthWellnessGuide.txt not found in data folder"
    return guide_path.read_text()

# Step 3: Create the agent with a research-focused system prompt
research_tools = [
    read_wellness_guide,
    write_todos,
    update_todo,
    list_todos,
]

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=research_tools,
    backend=filesystem_backend,
    system_prompt="""You are a Wellness Research Agent that produces structured, evidence-based reports.

When given a research task:
1. Create a todo list to plan and track your research process
2. Read the Health Wellness Guide using read_wellness_guide() for source material
3. Research the topic thoroughly, synthesizing evidence-based findings
4. Save your findings to a markdown file in the workspace with these sections:
   - # Title
   - ## Executive Summary
   - ## Strategies (one subsection per strategy with description, benefits, and implementation steps)
   - ## Quick Reference (summary table)
   - ## Sources
5. Update todo status as you complete each step
6. Provide a clear summary when complete

Be thorough and evidence-based. Cite information from the wellness guide when applicable."""
)

print("Research agent created!")

# Step 4: Test with the stress management research task
result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
    }]
})

print("\nResearch Agent response:")
print(result["messages"][-1].content)

print("\n" + "=" * 50)
print("\nTodo status:")
print(list_todos.invoke({}))


Research agent created!

Research Agent response:
Perfect! I have successfully completed the comprehensive stress management guide research project. Here's a summary of what I've accomplished:

## Project Summary

I have created a **Comprehensive Evidence-Based Stress Management Guide** that includes **7 detailed strategies** (exceeding your request for at least 5). The guide is saved as `/stress_management_comprehensive_guide.md` and includes:

### Key Features of the Guide:

**✅ Executive Summary** - Overview of stress impacts and strategy preview

**✅ Seven Evidence-Based Strategies:**
1. **Deep Breathing Techniques** - Box breathing for immediate relief
2. **Progressive Muscle Relaxation** - Systematic tension release
3. **Mindfulness Meditation** - Present-moment awareness training
4. **Regular Physical Exercise** - Natural stress hormone regulation
5. **Sleep Optimization** - Foundation for stress resilience
6. **Building Social Connections** - Emotional support networks
7. **Dig

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [16]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [17]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [19]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.

The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created your comprehensive morning routine guide for better energy. Here's what I accomplished:

## ✅ **Complete Morning Routine Guide Created**

I coordinated specialized research and content creation to deliver a comprehensive guide that includes:

### **🔬 Science-Based Foundation**
- Circadian rhythm research and energy optimization
- Evidence on morning light exposure and alertness
- Scientific backing for exercise timing and nutrition choices
- Psychology research on mindset practices

### **📋 Comprehensive Content Structure**
- **Exercise Section**: High, moderate, and low-impact options with optimal timing (6-8 AM window)
- **Nutrition Section**: The 40/30/30 formula with practical breakfast ideas and hydration protocols
- **Mindset Section**: Breathing exercises, gratitude practices, intention setting, and visualization techniques

### **⏰ Practical Implementation**
- **3 Sample Routines**: 60-minute complete, 30-minute express, 

In [20]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines and energy (completed)
✅ [todo_3] Create comprehensive morning routine guide (completed)
✅ [todo_5] Save the guide as a formatted markdown file (completed)

Generated files in workspace:
  [FILE] 2_week_exercise_schedule.txt (1202 bytes)
  [FILE] alex_comprehensive_wellness_program.md (8877 bytes)
  [FILE] alex_exercise_plan.md (3365 bytes)
  [FILE] alex_mindfulness_plan.md (9264 bytes)
  [FILE] alex_nutrition_plan.md (6020 bytes)
  [FILE] calming_elements.txt (672 bytes)
  [FILE] complete_morning_energy_guide_2024.md (10099 bytes)
  [FILE] comprehensive-morning-energy-guide.md (10119 bytes)
  [FILE] comprehensive_morning_routine_guide.md (15948 bytes)
  [FILE] comprehensive_stress_management_guide.md (12805 bytes)
  [FILE] cooldown_routine.txt (606 bytes)
  [FILE] energizing_morning_routine_guide.md (13159 bytes)
  [FILE] meal_plan_week_1.txt (3235 bytes)
  [FILE] meal_plan_week_2.txt (2875 bytes)
  [FILE] morn

## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [21]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [22]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.

    Args:
        user_id: The user's unique identifier

    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))

    if not items:
        return f"No profile found for {user_id}"

    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.

    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value

    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [23]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [22]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! Great to see you again. Based on your profile, I can see your primary goal is to improve energy levels and your secondary goal is better sleep. I also note you prefer morning exercise and like detailed information, plus you have mild anxiety to consider.

Here's a personalized exercise routine that aligns perfectly with your goals:

## **Morning Energy & Sleep Enhancement Routine**

### **Weekly Structure (6 days active, 1 rest)**

**Monday, Wednesday, Friday - Energizing Cardio**
- 20-30 minutes of moderate cardio (brisk walking, cycling, or swimming)
- These boost endorphins and energy without overstimulating your anxiety
- Best done 30-60 minutes after waking to kickstart your metabolism

**Tuesday, Thursday - Strength & Stability**
- 25-30 minutes of bodyweight or light resistance training
- Focus on compound movements: squats, modified push-ups, planks
- Builds muscle tone which improves metabolism and energy throughout the day

**Saturday - Active Recover

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [24]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [25]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [26]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.

    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans

    Args:
        skill_name: Name of the skill to load

    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"

    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [27]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [27]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Wellness Assessment Results

Based on your profile as a 35-year-old office worker, I've identified that your **sleep quality and physical activity** should be your primary focus areas, as these will have the most significant impact on your weight loss goal and overall wellness.

### Your Key Strengths:
- No major health conditions (excellent baseline)
- Vegetarian diet provides a healthy foundation
- Clear, achievable goals with strong motivation
- Prime age for implementing lasting lifestyle changes

### Priority Areas for Improvement:
1. **Sleep Quality** (Most Critical) - Poor sleep disrupts weight loss hormones
2. **Physical Activity** (Most Critical) - Counter sedentary work lifestyle  
3. **Nutritional Optimization** - Maximize your vegetarian diet for weight loss
4. **Workplace Wellness** - Integrate healthy habits into your work day

## Your Action Plan

### Immediate Actions (Start Today):
1. **Set a consistent bedtime routine** - Same sleep/wake times,

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [28]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.18


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [29]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [30]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [31]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
## 🎉 Your Complete 2-Week Wellness Program is Ready!

Alex, I've created a comprehensive wellness program tailored specifically for you, taking into account your vegetarian diet, mild anxiety, morning exercise preference, and goals to improve energy and sleep. Here's what you now have access to:

### 📁 Your Personal Wellness Files:
1. **`/alex_comprehensive_wellness_program.md`** - Your master program with daily schedules and integration strategies
2. **`/alex_exercise_plan.md`** - Complete 2-week workout plan (3x/week, 30 minutes, progressive)
3. **`/alex_nutrition_plan.md`** - Vegetarian meal plans focused on energy and sleep support
4. **`/alex_mindfulness_plan.md`** - Daily stress management and sleep optimization techniques

### 🎯 Program Highlights:

**Exercise**: Progressive 30-minute workouts (Mon/Wed/Fri) with confidence-building elements and anxiety-friendly approaches. Each workout includes warm-up, strength training, cardio, and calming cool-down pr

In [32]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Get exercise specialist to create 3x/week workout routine (completed)
✅ [todo_3] Get nutrition specialist to create vegetarian meal plan (completed)
✅ [todo_5] Get mindfulness specialist for stress/sleep optimization (completed)
✅ [todo_7] Create comprehensive wellness program document (completed)
✅ [todo_9] Save exercise plan to file (completed)
✅ [todo_11] Save nutrition plan to file (completed)
✅ [todo_13] Save mindfulness plan to file (completed)
✅ [todo_15] Save integrated program overview to file (completed)

GENERATED FILES
  [FILE] 2_week_exercise_schedule.txt (1202 bytes)
  [FILE] alex_comprehensive_wellness_program.md (8877 bytes)
  [FILE] alex_exercise_plan.md (3365 bytes)
  [FILE] alex_mindfulness_plan.md (9264 bytes)
  [FILE] alex_nutrition_plan.md (6020 bytes)
  [FILE] calming_elements.txt (672 bytes)
  [FILE] comprehensive-morning-energy-guide.md (10119 bytes)
  [FILE] comprehensive_morning_routine_guide.md (15948 bytes)
  [FILE] cooldown_rou

In [33]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of sleep_improvement_research.md:
# Evidence-Based Sleep Improvement Strategies: Comprehensive Research Summary

## Executive Summary
This research summary addresses three specific sleep issues through evidence-based interventions:
1. Inconsistent bedtime schedule (10pm-1am range)
2. Phone/screen use in bed
3. Morning fatigue despite adequate sleep duration

All recommendations are grounded in peer-reviewed sleep science research and organized with implementation timelines and expected outcomes.

## Problem Analysis

### Issue 1: Inconsistent Bedtime Schedule (3-hour window)
**Scientific Impact:**
- Disrupts circadian rhythm synchronization
- Reduces sleep efficiency and quality
- Affects melatonin production timing
- Creates "social jet lag" effect

### Issue 2: Screen Use in Bed
**Scientific Impact:**
- Blue light suppresses melatonin production by 23-48% (Harvard studies)
- Increases cortisol levels before sleep
- Stimulates cognitive arousal
- Delays sleep onset by 10-60 

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
Subagents should share file tools that are used to update shared context - for example, when updating todos (luckily this automatically shared via the backend with deepagents). Subagents should have distinct tools related to their purpose (i.e. a nutrition specialist subagent should be able to research healthy foods, while this tool would be irrelevant for a fitness specialist subagent).

Model selection for subagents should be based on how complex the subagent's purpose is - simpler tasks should use more cost-efficient and faster models, while more complex tasks should use more powerful models. This will help to optimize costs and speeds when using multiple subagents. Additionally, user-facing output may choose to use a model that performs better for writing.

Subagents should be granular enough to have a distinct purpose but not too specialized that they are never used. For example, it could be helpful to have a subagent that can select healthy food options, but specializing to healthy vegetable options would likely be overkill. Having too many distinct subagents will add complexity for the coordinator and should be avoided.

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
A production app would need safety guardrails, such as the note to "consult a physician" that we've added to previous agents. Additionally, persistent storage would be a must to avoid losing all data every time the server restarts - we could use PostgresStore for this as mentioned in the notebook. Additionally, multi-user support would likely require creating unique namespaces and workspaces for each user. Additionally, adding LangSmith would allow us to monitor our app & track performance and errors. In order to manage costs, we could limit how many subagents are used, what model each subagent uses (i.e. cost-effective models for simpler tasks), and limit each user's usage to a certain amount to avoid costs above our budget.

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [33]:
### YOUR CODE HERE ###

# Step 1: Define your subagent configurations
fitness_coach = {
    "name": "fitness-coach",
    "description": "Expert in exercise programming and 30-day fitness progressions. Use for creating workout plans, daily exercise challenges, and fitness milestones.",
    "system_prompt": """You are a fitness coach specializing in 30-day challenge programs.

Your expertise includes:
- Progressive workout programming that builds week over week
- Bodyweight and minimal-equipment exercises
- Daily exercise challenges that are achievable but motivating
- Adapting difficulty based on user feedback and energy levels

Always consider the user's fitness level, time constraints, and any physical limitations.
Structure plans in weekly phases with clear daily tasks.
When adapting plans, explain what changed and why.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_coach = {
    "name": "mindfulness-coach",
    "description": "Expert in stress management, sleep optimization, meditation, and mental wellness habits. Use for mindfulness challenges, sleep routines, and stress reduction plans.",
    "system_prompt": """You are a mindfulness and mental wellness coach specializing in 30-day habit-building programs.

Your expertise includes:
- Progressive meditation and breathing exercises
- Sleep hygiene optimization and evening routines
- Daily mindfulness challenges for stress reduction
- Building sustainable mental wellness habits over 30 days

Be supportive and non-judgmental.
Structure plans in weekly phases with clear daily practices.
When adapting plans, reference the user's check-in data to personalize adjustments.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Subagent configurations defined: fitness-coach, mindfulness-coach")


# Step 2: Create any additional tools you need

# Daily check-in storage for tracking progress
CHECKIN_LOG = []

@tool
def log_daily_checkin(
    user_id: str,
    day: int,
    mood: str,
    energy_level: str,
    completed_tasks: str,
    notes: str
) -> str:
    """Log a daily wellness check-in for progress tracking.

    Args:
        user_id: The user's unique identifier
        day: Day number in the 30-day challenge (1-30)
        mood: User's mood rating (great/good/okay/low)
        energy_level: User's energy level (high/medium/low)
        completed_tasks: Comma-separated list of completed daily tasks
        notes: Any additional notes or feedback from the user

    Returns:
        Confirmation message
    """
    entry = {
        "user_id": user_id,
        "day": day,
        "mood": mood,
        "energy_level": energy_level,
        "completed_tasks": completed_tasks,
        "notes": notes,
    }
    CHECKIN_LOG.append(entry)
    return f"Check-in logged for {user_id}, Day {day}. Mood: {mood}, Energy: {energy_level}"

@tool
def get_checkin_history(user_id: str) -> str:
    """Retrieve all daily check-in entries for a user.

    Args:
        user_id: The user's unique identifier

    Returns:
        Formatted check-in history
    """
    user_entries = [e for e in CHECKIN_LOG if e["user_id"] == user_id]
    if not user_entries:
        return f"No check-ins found for {user_id}"

    result = [f"Check-in history for {user_id} ({len(user_entries)} entries):"]
    for entry in user_entries:
        result.append(
            f"  Day {entry['day']}: Mood={entry['mood']}, Energy={entry['energy_level']}, "
            f"Completed=[{entry['completed_tasks']}], Notes: {entry['notes']}"
        )
    return "\n".join(result)

print("Check-in tools defined: log_daily_checkin, get_checkin_history")


# Step 3: Build the main coordinator agent
TODO_STORE.clear()
CHECKIN_LOG.clear()

challenge_tools = [
    # Planning
    write_todos,
    update_todo,
    list_todos,
    # Long-term Memory
    get_user_profile,
    save_user_preference,
    # Daily Check-ins (Context Management)
    log_daily_checkin,
    get_checkin_history,
]

wellness_challenge_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=challenge_tools,
    backend=filesystem_backend,  # File ops sandboxed to workspace
    subagents=[fitness_coach, mindfulness_coach],
    system_prompt="""You are a 30-Day Wellness Challenge Coach that creates and manages personalized wellness programs.

## Your Role
- Design personalized 30-day wellness challenges with weekly phases
- Track daily progress through check-ins
- Adapt recommendations based on user feedback and check-in trends
- Generate weekly summary reports saved to files

## Workflow
1. **Profile Review**: Check the user's profile for goals, conditions, and preferences
2. **Challenge Planning**: Create a todo list for the 30-day challenge phases
3. **Plan Generation**: Delegate to specialists:
   - fitness-coach: Exercise challenges, workout progressions, and daily fitness tasks
   - mindfulness-coach: Mindfulness exercises, sleep routines, and stress management
4. **Save Plans**: Save the full 30-day challenge plan to a markdown file
5. **Daily Check-ins**: Log progress using log_daily_checkin, track mood and energy
6. **Weekly Adaptation**: Review check-in history, adapt the plan, and save weekly summary reports

## 30-Day Challenge Structure
- **Week 1 (Days 1-7)**: Foundation - Build basic habits
- **Week 2 (Days 8-14)**: Growth - Increase intensity and add variety
- **Week 3 (Days 15-21)**: Challenge - Push boundaries with harder goals
- **Week 4 (Days 22-30)**: Mastery - Consolidate habits for long-term sustainability

## Important
- Always check user profile first for medical conditions and dietary restrictions
- Save the 30-day plan and weekly summary reports as separate markdown files
- When adapting, reference specific check-in data to explain changes
- Be encouraging and celebrate progress"""
)

print("30-Day Wellness Challenge Coach created with all 4 Deep Agent elements!")


# Step 4: Test with a user creating their 30-day challenge
result = wellness_challenge_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I want to start a 30-day wellness challenge!

I'd like to focus on:
1. Daily exercise that starts easy and builds up over the month
2. A mindfulness and sleep improvement routine

Please create a personalized 30-day challenge plan and save it to a file."""
    }]
})

print("\nWellness Challenge Coach response:")
print(result["messages"][-1].content)

print("\n" + "=" * 50)
print("\nTodo status:")
print(list_todos.invoke({}))


# Step 5: Simulate a daily check-in and adaptation

# Simulate a few days of check-ins to build history
log_daily_checkin.invoke({
    "user_id": "user_alex", "day": 1, "mood": "good", "energy_level": "medium",
    "completed_tasks": "morning walk, 5-min meditation",
    "notes": "Good start but meditation was hard to focus"
})
log_daily_checkin.invoke({
    "user_id": "user_alex", "day": 2, "mood": "good", "energy_level": "medium",
    "completed_tasks": "morning walk, 5-min meditation, evening stretching",
    "notes": "Feeling more settled, sleep was better"
})
log_daily_checkin.invoke({
    "user_id": "user_alex", "day": 3, "mood": "okay", "energy_level": "low",
    "completed_tasks": "morning walk",
    "notes": "Stressful day at work, skipped meditation and stretching"
})

print("\n" + "=" * 50)
print("\nCheck-in history before adaptation:")
print(get_checkin_history.invoke({"user_id": "user_alex"}))

# Now ask the coach to review progress and adapt
TODO_STORE.clear()

adaptation_result = wellness_challenge_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """My user_id is user_alex. I'm on Day 3 of my 30-day challenge.

Can you review my check-in history, adapt my plan based on my feedback,
and save a weekly progress report?

I've been struggling with the meditation - it's hard to focus. Also, Day 3 was
rough because of work stress. I need something more manageable for busy days."""
    }]
})

print("\n" + "=" * 50)
print("\nAdaptation response:")
print(adaptation_result["messages"][-1].content)

print("\n" + "=" * 50)
print("\nFinal todo status:")
print(list_todos.invoke({}))


Subagent configurations defined: fitness-coach, mindfulness-coach
Check-in tools defined: log_daily_checkin, get_checkin_history
30-Day Wellness Challenge Coach created with all 4 Deep Agent elements!

Wellness Challenge Coach response:
Each day, I'll help you log:
- **Mood & Energy levels** (track improvements over time)
- **Completed tasks** (both fitness and mindfulness)
- **Personal notes** (challenges, victories, observations)

At the end of each week, I'll create a progress report and adapt the following week based on your feedback and patterns.

## 🚀 Ready to Start?

Your complete 30-day plan is saved in `/30_day_wellness_challenge_alex.md` - you can refer to it anytime!

**When would you like to begin Day 1?** Just let me know, and I'll guide you through:
- Day 1 tasks: 10-min easy walk + gentle stretching + 5-min mindful breathing
- Your first daily check-in
- Any questions about getting started

This challenge is designed specifically for your goals of improving energy and sl

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)